In [1]:
!pip install google-adk requests --quiet

In [2]:
import os
import requests
from typing import Optional, List, Dict

import vertexai
from google.adk.agents import Agent
from google.adk.runners import InMemoryRunner
from google.genai import types

PROJECT_ID = "qwiklabs-gcp-00-11b9d8ae6979"
LOCATION = "us-central1"

vertexai.init(project=PROJECT_ID, location=LOCATION)

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"

In [3]:
from google.cloud import secretmanager

def get_secret(secret_id: str) -> str:
    """Retrieve a secret value from Google Secret Manager.

    Args:
        secret_id (str): The name of the secret to retrieve.

    Returns:
        str: The decoded secret string value.
    """
    sm_client = secretmanager.SecretManagerServiceClient()
    name = f"projects/{PROJECT_ID}/secrets/{secret_id}/versions/latest"
    response = sm_client.access_secret_version(request={"name": name})
    return response.payload.data.decode("UTF-8")

GOOGLE_MAPS_API_KEY = get_secret("GOOGLE_MAPS_API_KEY")
print("API key loaded from Secret Manager")

API key loaded from Secret Manager


In [4]:
def get_lat_lon(city: str, state: str) -> Optional[Dict[str, float]]:
    """Use the Google Maps Geocoding API to convert a city and state to latitude and longitude.

    Args:
        city (str): The name of the city (e.g., "Austin").
        state (str): The two-letter state abbreviation (e.g., "TX").

    Returns:
        Optional[Dict[str, float]]: A dictionary with 'lat' and 'lon' keys,
        or None if the location could not be geocoded.
    """
    address = f"{city}, {state}"
    url = "https://maps.googleapis.com/maps/api/geocode/json"
    params = {"address": address, "key": GOOGLE_MAPS_API_KEY}

    response = requests.get(url, params=params)
    if response.status_code != 200:
        print(f"Geocoding API error: {response.status_code}")
        return None

    data = response.json()
    if data.get("status") != "OK" or not data.get("results"):
        print(f"No geocoding results for: {address}")
        return None

    location = data["results"][0]["geometry"]["location"]
    return {"lat": location["lat"], "lon": location["lng"]}

In [5]:
def get_extended_weather_forecast(lat: float, lon: float) -> Optional[List[Dict[str, str]]]:
    """Fetch the extended weather forecast from the U.S. National Weather Service (NWS) API
    based on a given latitude and longitude.

    Args:
        lat (float): Latitude of the location (e.g., 38.8977).
        lon (float): Longitude of the location (e.g., -77.0365).

    Returns:
        Optional[List[Dict[str, str]]]: A list of forecast dictionaries for each time period,
        each containing:
            - 'name': Name of the forecast period (e.g., "Today", "Tonight")
            - 'startTime': ISO timestamp for the start of the forecast period
            - 'temperature': Temperature value
            - 'temperatureUnit': Temperature unit (e.g., "F" or "C")
            - 'windSpeed': Wind speed description
            - 'windDirection': Wind direction (e.g., "NW")
            - 'shortForecast': Short summary (e.g., "Partly Sunny")
            - 'detailedForecast': Full text forecast description
        Returns None if the forecast could not be retrieved.
    """
    headers = {
        "User-Agent": "MyWeatherApp (student-00-eab28a1c4614@qwiklabs.net)",
        "Accept": "application/geo+json"
    }

    # Step 1: Get metadata to find the forecast URL
    points_url = f"https://api.weather.gov/points/{lat},{lon}"
    response = requests.get(points_url, headers=headers)

    if response.status_code != 200:
        print(f"Error fetching data from points endpoint: {response.status_code}")
        return None

    points_data = response.json()
    forecast_url = points_data["properties"].get("forecast")

    if not forecast_url:
        print("Forecast URL not found in response.")
        return None

    # Step 2: Fetch the forecast data
    forecast_response = requests.get(forecast_url, headers=headers)
    if forecast_response.status_code != 200:
        print(f"Error fetching forecast: {forecast_response.status_code}")
        return None

    forecast_data = forecast_response.json()
    periods = forecast_data["properties"].get("periods", [])

    return [
        {
            "name": p.get("name", ""),
            "startTime": p.get("startTime", ""),
            "temperature": str(p.get("temperature", "")),
            "temperatureUnit": p.get("temperatureUnit", ""),
            "windSpeed": p.get("windSpeed", ""),
            "windDirection": p.get("windDirection", ""),
            "shortForecast": p.get("shortForecast", ""),
            "detailedForecast": p.get("detailedForecast", ""),
        }
        for p in periods
    ]

In [6]:
WEATHER_AGENT_INSTRUCTIONS = """
You are Mr. Weather, a friendly and knowledgeable weather assistant.

When a user asks about the weather in a city:
1. Use the get_lat_lon tool to convert the city and state to latitude and longitude.
2. Use the get_extended_weather_forecast tool with those coordinates to get the forecast.
3. Summarize the current conditions and highlight any weather alerts or notable conditions
   (e.g., extreme heat, storms, heavy wind, snow).
4. Always mention the city name, temperature, and a brief outlook for the next day or two.
5. If conditions are severe or dangerous, clearly flag this as a WEATHER ALERT.

Be concise, friendly, and helpful. Use plain language, not raw data dumps.
If a tool fails or returns no data, let the user know politely.
"""

In [7]:
weather_agent_gemini = Agent(
    name="Pat",
    model="gemini-2.0-flash",
    description="Mr. Weather, the friendly weather agent.",
    instruction=WEATHER_AGENT_INSTRUCTIONS,
    tools=[get_extended_weather_forecast, get_lat_lon],
)

In [8]:
from google.adk.runners import InMemoryRunner
from google.genai import types

async def generate_weather_message(user_input: str, agent: Agent = weather_agent_gemini) -> str:
    """Send a message to the weather agent and return its response.

    Args:
        user_input (str): The user's weather query.
        agent (Agent): The ADK agent to use. Defaults to Gemini agent.

    Returns:
        str: The agent's response text.
    """
    runner = InMemoryRunner(agent=agent, app_name="weather_app")
    session = await runner.session_service.create_session(
        app_name="weather_app", user_id="user_01"
    )

    message = types.Content(
        role="user", parts=[types.Part(text=user_input)]
    )

    response_text = ""
    for event in runner.run(
        user_id="user_01",
        session_id=session.id,
        new_message=message
    ):
        if event.is_final_response() and event.content:
            for part in event.content.parts:
                if part.text:
                    response_text += part.text

    return response_text

In [9]:
from IPython.display import Markdown, display

test_cities = [
    "What's the weather in Austin, TX?",
    "Give me a weather report for Miami, FL",
    "Any weather alerts for Chicago, IL?",
    "What should I expect weather-wise in Seattle, WA this week?",
]

print("=" * 60)
print("BEGIN WEATHER AGENT TEST — Various US Cities")
print("=" * 60)

for query in test_cities:
    print(f"\nYou: {query}")
    response = await generate_weather_message(query)
    display(Markdown(f"**Pat:** {response}"))
    print("-" * 60)

BEGIN WEATHER AGENT TEST — Various US Cities

You: What's the weather in Austin, TX?


**Pat:** The weather in Austin, TX is partly sunny with a high near 85°F today. There is a slight chance of rain showers. Winds will be 10 to 15 mph, with gusts as high as 30 mph.

The forecast for tomorrow is mostly cloudy with a slight chance of rain showers and thunderstorms and a high near 84°F. Saturday has a 90% chance of showers and thunderstorms.


------------------------------------------------------------

You: Give me a weather report for Miami, FL


**Pat:** The weather in Miami, FL is mostly sunny with a high near 80°F today and a slight chance of rain showers. The wind will be around 15 mph, with gusts as high as 21 mph.

The outlook for Friday is mostly sunny with a high near 80°F and a slight chance of rain showers.

------------------------------------------------------------

You: Any weather alerts for Chicago, IL?


**Pat:** Currently, Chicago is experiencing widespread fog with a temperature of 39°F. There's a chance of rain showers before noon.

Looking ahead, Friday brings a chance of showers and thunderstorms, especially between 9am and noon, with a high near 67°F and gusts up to 30 mph. Be aware of possible strong winds.

------------------------------------------------------------

You: What should I expect weather-wise in Seattle, WA this week?


**Pat:** This week in Seattle, expect mostly cloudy conditions with a high near 50°F today. There's a chance of rain today and a high likelihood of rain on Friday and Saturday. Sunday night through Tuesday night, there is a chance of rain and snow.

------------------------------------------------------------


In [10]:
from IPython.display import Markdown, display

print("Welcome to Ultra Weather! Type 'exit' to end the conversation.")

while True:
    user_input = input("You: ")
    if user_input.lower() in ["bye", "exit", "quit"]:
        print("Chatbot: Goodbye!")
        break
    else:
        response = f"Chatbot: {await generate_weather_message(user_input)}"
        display(Markdown(response))

Welcome to Ultra Weather! Type 'exit' to end the conversation.
You: How is the weather in Boise Idaho right now?


Chatbot: Good morning! In Boise, Idaho, it's partly sunny with a slight chance of rain before 5pm. The temperature is expected to reach 49°F, with a northwest wind of 10 to 17 mph and gusts up to 26 mph. Tomorrow will be partly sunny with a high near 50°F. Have a great day!


You: How is the weather in Springfield IL? Do I need to wear a hat and a coat?


Chatbot: Good morning! The weather in Springfield, IL is mostly cloudy with a high near 61°F today. There may be patchy fog before noon. Tonight, there's a chance of light rain and patchy fog with a low around 51°F.

Friday will be warmer, with a high near 78°F, but there's also a chance of thunderstorms and rain. The wind will be quite strong, with gusts as high as 35 mph.

Given the temperatures, you might not need a heavy coat today, but a hat could be useful, especially later in the week with the chance of rain and thunderstorms.


You: What is the weather like in Englewood Florida?


Exception in thread Thread-17 (_asyncio_thread_main):
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/google/adk/models/google_llm.py", line 241, in generate_content_async
    response = await self.api_client.aio.models.generate_content(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/google/genai/models.py", line 7030, in generate_content
    return await self._generate_content(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/google/genai/models.py", line 5848, in _generate_content
    response = await self._api_client.async_request(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/google/genai/_api_client.py", line 1432, in async_request
    result = await self._async_request(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/google/genai/_api_client.py", 

Chatbot: 

You: exit
Chatbot: Goodbye!
